# Sacred Speech Scenario Planner â€” v4 (wizard)

**Version 4.0 (preview)** Â· Distributed via GitHub Â· Each user supplies their own OpenAI API key.

A guided 5-screen wizard for planning context-appropriate sacred speech for any Army scenario. No code reading required.

## Open in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KirtisChristensen/sacred-speech-planner/blob/v4-wizard/Sacred_Speech_Planner_v4.ipynb)

## How to use

1. **Add your OpenAI API key** (once per Google account). In Colab: ðŸ”‘ icon â†’ **+ Add new secret** â†’ name `OPENAI_API_KEY`, paste value, toggle **Notebook access ON**. Get a key at <https://platform.openai.com/api-keys>.
2. **Runtime â†’ Run all** (one time). First time only, click **Grant access** on any consent popup, then **Run all** again.
3. Scroll to the wizard below the second cell. Use **Next â–¶** / **â—€ Back** to move between screens.

## The 5 screens

| # | Screen | What you do |
|---|---|---|
| 1 | Welcome | Confirm key works (automatic). |
| 2 | Describe | Paste a plain-language scenario â†’ click **Autofill from description**. |
| 3 | Review | Edit any form fields the AI got wrong. |
| 4 | Plan | Read the generated plan; (optional) add your concept, scripture, illustrations. |
| 5 | Draft | Click **Generate AI Draft**; export the result to Markdown. |

## OPSEC

Generic descriptors only â€” no PII, names, unit identifiers, or sensitive operational detail. Inputs in screens 2 and 4 are sent to the OpenAI API.

## Pluralism

Descriptive and culturally adaptive. Does not prescribe theology. The chaplain supplies all sacred-text content.

In [ ]:
# Sacred Speech Planner v4 — wizard UI
# Setup cell: imports, LLM client, profiler, prompts, widgets.
# Run this cell once. Then run the next cell to launch the wizard.

import os, json, datetime
from dataclasses import dataclass
from typing import List

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Colab Secrets bridge (no-op outside Colab)
try:
    from google.colab import userdata  # type: ignore
    for _k in ('OPENAI_API_KEY', 'XAI_API_KEY', 'LLM_PROVIDER', 'LLM_MODEL'):
        try:
            _v = userdata.get(_k)
            if _v and not os.environ.get(_k):
                os.environ[_k] = _v
        except Exception:
            pass
except Exception:
    pass

try:
    from openai import OpenAI
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])
    from openai import OpenAI

PROVIDER = os.environ.get('LLM_PROVIDER', 'openai').lower()
if PROVIDER == 'xai':
    API_KEY  = os.environ.get('XAI_API_KEY', '')
    BASE_URL = 'https://api.x.ai/v1'
    MODEL    = os.environ.get('LLM_MODEL', 'grok-4')
else:
    PROVIDER = 'openai'
    API_KEY  = os.environ.get('OPENAI_API_KEY', '')
    BASE_URL = 'https://api.openai.com/v1'
    MODEL    = os.environ.get('LLM_MODEL', 'gpt-4o-mini')

client = OpenAI(api_key=(API_KEY or 'placeholder'), base_url=BASE_URL)

def llm_chat(system, user, temperature=0.4, response_format=None):
    kwargs = dict(model=MODEL,
                  messages=[{'role':'system','content':system},
                            {'role':'user','content':user}],
                  temperature=temperature)
    if response_format is not None:
        kwargs['response_format'] = response_format
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content

# ----- Profiler (rules-based) -----
@dataclass
class ScenarioInputs:
    event_type: str; event_other: str; situation: str; location: str; optempo: str
    audience_primary: str; audience_size: str; trauma: str; fatigue: str
    orientation: str; cultural_mix: str; time: str; needs: List[str]
    tradition: str; text_pref: str

@dataclass
class ScenarioProfile:
    summary: str; recommended_form: str; target_length: str
    structure: List[str]; tone: List[str]
    trauma_posture: List[str]; cultural_posture: List[str]
    delivery_tips: List[str]; avoid: List[str]

def _is_high(level):
    return level in ('High', 'Acute / immediate', 'Severe (sustained ops)')

def _time_minutes(t):
    return {'3-5 min':3,'5-10 min':5,'10-15 min':10,'15-20 min':15,'20+ min':20}.get(t, 7)

def profile(inp):
    minutes = _time_minutes(inp.time)
    high_trauma = _is_high(inp.trauma)
    high_fatigue = _is_high(inp.fatigue)
    pluralistic = inp.cultural_mix in ('Moderately mixed', 'Highly pluralistic')
    emotional = inp.orientation == 'Mostly emotional'
    et = inp.event_type
    if et == 'Memorial service':
        form = 'Memorial homily — Lament + Hope arc'
    elif et == 'Graveside remarks':
        form = 'Graveside remarks — brief, dignified, pastoral'
    elif et == 'Post-trauma debrief / response':
        form = 'Trauma-responsive presence message (Acknowledge–Pivot–Hope)'
    elif et == 'Disaster recovery worship':
        form = 'Short field devotional — trauma-responsive hope message'
    elif et == 'Deployment send-off':
        form = 'Send-off charge — courage + connection + blessing'
    elif et == 'Redeployment / homecoming':
        form = 'Homecoming reflection — gratitude + reintegration'
    elif et == 'Daily "word of the day"':
        form = 'Micro-devotional — single image, single point'
    elif et == 'Unit resilience / training event':
        form = 'Resilience teaching — story + principle + application'
    elif et == 'Garrison chapel service':
        form = 'Standard homily — narrative arc with one clear point'
    else:
        form = 'Adaptive short-form message (Acknowledge–Pivot–Hope)'
    if high_trauma and minutes <= 10 and 'devotional' not in form.lower() and 'graveside' not in form.lower():
        form += ' [trauma-responsive, ultra-concise variant]'
    if high_trauma or et in ('Post-trauma debrief / response','Disaster recovery worship','Memorial service','Graveside remarks'):
        a = max(1, round(minutes*0.25)); c = max(1, round(minutes*0.45)); h = max(1, minutes-a-c)
        structure = [f'1. Acknowledge (~{a} min) — name the loss/exhaustion/reality without minimizing.',
                     f'2. Anchor (~{c} min) — one short sacred-text or story anchor; one clear idea.',
                     f'3. Hope / Blessing (~{h} min) — transcendent pivot, presence, and closing blessing.']
    elif et == 'Daily "word of the day"':
        structure = ['1. One image or phrase (~30 sec).','2. One sentence of meaning (~1 min).','3. One sending line (~30 sec).']
    else:
        i = max(1, round(minutes*0.15)); b = max(1, round(minutes*0.55)); c = max(1, minutes-i-b)
        structure = [f'1. Hook / context (~{i} min) — meet the audience where they are.',
                     f'2. Body (~{b} min) — one main point, one story, one sacred-text anchor.',
                     f'3. Call / blessing (~{c} min) — clear application or sending.']
    tone = []
    if high_trauma: tone += ['Empathetic','Lament + hope','Calm, slow, grounded']
    else: tone += ['Hope-focused','Warm and direct']
    if emotional: tone.append('Heart-forward (image and story over argument)')
    elif inp.orientation == 'Mostly cognitive': tone.append('Clear, structured, one explicit point')
    else: tone.append('Balanced head/heart')
    if high_fatigue: tone.append('Low cognitive load — short sentences, repeated key phrase')
    if high_trauma:
        trauma_posture = ['Validate suffering before pivoting to hope.',
                          'Use lament language; avoid platitudes.',
                          'Emphasize presence ("you are not alone") over explanation.',
                          'No graphic re-description of the event.',
                          'Offer a concrete next-step image (small, doable).']
    else:
        trauma_posture = ['Stay audience-centered; assume some carry hidden weight.',
                          'Avoid platitudes; keep hope grounded in real action or presence.']
    if pluralistic:
        cultural_posture = ['Speak from your tradition without requiring shared belief.',
                            'Use universal human anchors (breath, light, presence, courage) alongside sacred text.',
                            'Avoid in-group shorthand or assumed doctrinal terms.',
                            'Offer blessing in inclusive form; let listeners receive it from their own tradition.']
    else:
        cultural_posture = ['You may use tradition-specific language with care; still honor any outliers present.']
    delivery_tips = ['Short sentences. Plain language. No jargon.',
                     'Slow pace; deliberate pauses after key lines.',
                     'Eye contact across the formation, not just the front row.',
                     f'Hard time cap: {minutes} min. Cut content, not pace.']
    if inp.location in ('Field / outdoors','Motor pool','Disaster site'):
        delivery_tips.append('Project voice; assume wind/equipment noise; no notes-shuffling.')
    if inp.audience_size == 'Large formation (100+)':
        delivery_tips.append('Larger gestures; one clear repeated phrase to anchor the back rows.')
    if high_fatigue:
        delivery_tips.append('Open with permission to sit/rest; do not require participation.')
    avoid = []
    if high_trauma:
        avoid += ['Triumphalism or quick resolution of grief.',
                  'Detailed re-narration of the traumatic event.',
                  'Calls to action that demand emotional output.']
    if pluralistic: avoid.append('Insider phrases that require shared doctrine to land.')
    if minutes <= 10: avoid.append('Multi-point sermons; pick ONE idea.')
    if high_fatigue: avoid.append('Long preambles, throat-clearing, or thanking lists.')
    descriptors = []
    if high_trauma: descriptors.append('high-trauma')
    if high_fatigue: descriptors.append('high-fatigue')
    descriptors.append(f'{minutes}-min')
    descriptors.append(inp.optempo.lower())
    summary = (f"This is a **{', '.join(descriptors)}** event — "
               f"{inp.audience_primary.lower()} at a {inp.location.lower()} "
               f"during {inp.situation.lower()}. "
               f"Cultural mix: {inp.cultural_mix.lower()}. "
               f"Audience need(s): {', '.join(inp.needs) if inp.needs else '(unspecified)'}.")
    return ScenarioProfile(summary=summary, recommended_form=form, target_length=inp.time,
                           structure=structure, tone=tone, trauma_posture=trauma_posture,
                           cultural_posture=cultural_posture, delivery_tips=delivery_tips, avoid=avoid)

def render_profile(p, inp):
    def b(items): return '\n'.join(f'- {x}' for x in items) if items else '- (none)'
    text_anchor = inp.text_pref if inp.text_pref else '_(chaplain selects)_'
    tradition = inp.tradition if inp.tradition else '_(not specified)_'
    return f"""## Scenario Profile Summary
{p.summary}

## Recommended Form
**{p.recommended_form}**  
Target length: **{p.target_length}**

### Structure & Timing
{b(p.structure)}

### Tone
{b(p.tone)}

### Trauma-Responsive Posture
{b(p.trauma_posture)}

### Cultural / Pluralism Posture
{b(p.cultural_posture)}

### Avoid
{b(p.avoid)}

## Sacred Speech Organizer (fillable)
```
Title / Theme:

Scripture / Sacred Text Anchor: {text_anchor}

Context Acknowledgment:

Core Message (ONE clear point):

Illustration / Story:

Hope / Transcendent Pivot:

Call to Action or Closing Blessing:

Chaplain tradition note (internal): {tradition}
```

## Delivery Tips
{b(p.delivery_tips)}
"""

# ----- LLM prompts -----
AUTOFILL_SYSTEM = """You are an assistant helping a U.S. Army chaplain plan sacred speech.
Read the chaplain's free-form scenario description and produce a JSON object that
maps onto a structured form. Pick the BEST option from each allowed list.
If a field is genuinely unspecified, use the provided default.
Do NOT invent PII or unit identifiers. Output JSON only — no prose, no markdown.

Required JSON keys and allowed values:

- event_type: one of ["Field devotional","Memorial service","Graveside remarks",
  "Post-trauma debrief / response","Disaster recovery worship","Deployment send-off",
  "Redeployment / homecoming","Daily \\"word of the day\\"",
  "Unit resilience / training event","Garrison chapel service","Other"]
- event_other: free text (only if event_type == "Other"; else "")
- situation: one of ["Natural disaster","Combat loss","Training accident",
  "Routine resilience","Deployment cycle","Suicide / line-of-duty death","Other"]
- location: one of ["Field / outdoors","Motor pool","Chapel tent","Disaster site",
  "Briefing room","Garrison chapel","Other"]
- optempo: one of ["Immediate response","Sustained operations","Recovery phase",
  "Garrison / steady state"]
- audience_primary: one of ["Soldiers in active recovery / response",
  "Exhausted work crews","Bereaved family / next of kin","Mixed unit formation",
  "Leadership / command team","Families and dependents","Trainees / initial entry",
  "Mixed civilian + military"]
- audience_size: one of ["Small (<20)","Medium (20-100)","Large formation (100+)"]
- trauma: one of ["Low","Medium","High","Acute / immediate"]
- fatigue: one of ["Low","Medium","High","Severe (sustained ops)"]
- orientation: one of ["Mostly emotional","Balanced","Mostly cognitive"]
- cultural_mix: one of ["Homogeneous","Moderately mixed","Highly pluralistic"]
- time: one of ["3-5 min","5-10 min","10-15 min","15-20 min","20+ min"]
- needs: array of zero or more from ["Comfort","Encouragement","Hope",
  "Lament / processing grief","Meaning-making","Resilience","Communal bonding",
  "Individual reflection","Challenge / call to action"]
- text_pref: free text scripture/text preference if explicitly mentioned, else ""
- tradition: free text chaplain tradition if explicitly mentioned, else ""
- rationale: 1-3 short bullets explaining your key choices (string with line breaks).

Defaults if truly unspecified: event_type="Field devotional", situation="Other",
location="Field / outdoors", optempo="Garrison / steady state",
audience_primary="Mixed unit formation", audience_size="Small (<20)",
trauma="Low", fatigue="Low", orientation="Balanced", cultural_mix="Highly pluralistic",
time="5-10 min", needs=["Encouragement","Hope"], text_pref="", tradition="".
"""

DRAFT_SYSTEM_PROMPT = """You are an assistant helping a U.S. Army chaplain shape SACRED SPEECH (devotionals, homilies, memorial remarks, blessings) for a specific scenario.

You will receive THREE inputs:
A) A SCENARIO PROFILE (form, structure, tone, trauma/pluralism posture, time cap).
B) Optional CHAPLAIN CONTENT — concept, scripture/texts, illustrations, draft, constraints.
C) Hard rules below.

PRIORITY OF SOURCES (highest first):
1. Chaplain Content. Treat it as authoritative. Do NOT replace the chaplain's scripture choices, illustrations, core message, or wording with your own preferences. Tighten, fit to time, place into structure, polish — do not overrule.
2. Scenario Profile. Use it to shape structure, tone, length, and posture.
3. Your own additions. Only fill gaps the chaplain left blank, and clearly mark them as suggestions in the "Notes for the Chaplain" section.

HARD RULES — never violate:
1. Pluralism-safe. Inclusive language for a mixed military formation. Do NOT prescribe doctrine. If the chaplain provided tradition-specific text or wording, keep it; do not water it down — but frame it so listeners from other traditions can still receive the message with respect.
2. Trauma-responsive when indicated. Validate suffering before pivoting to hope. No platitudes ("everything happens for a reason," "be strong," "God needed another angel"). No graphic re-narration of traumatic events. Emphasize presence over explanation.
3. Time discipline. Stay within the chaplain's time cap (~130 words/min spoken, then trim 20% for pauses). Prefer ONE clear core idea.
4. Plain language. Short sentences. No jargon or insider shorthand that requires shared belief.
5. Audience respect. Never demand emotional output from exhausted, grieving, or traumatized listeners.
6. No PII. Do not invent names, units, locations, or identifying detail. Use only generic descriptors provided.
7. Draft only. This is a starting draft for the chaplain to edit. Do not claim authority.
8. Scripture handling. If the chaplain provided full scripture text, you may quote it. If only a reference, paraphrase carefully and note "verify exact wording" in the chaplain notes — do NOT fabricate quotations.

OUTPUT FORMAT (Markdown), in this exact order, with these headings:

### Title / Theme (draft)
### Context Acknowledgment (draft)
### Core Message — one sentence (draft)
### Scripture / Sacred Text Anchor (draft)
### Illustration / Story (short, non-graphic) (draft)
### Hope / Transcendent Pivot (draft)
### Closing Blessing (draft)
### Notes for the Chaplain
- Bullets: which sections came from chaplain content vs. were filled in by you, what to verify (especially scripture quotations), where to insert tradition-specific content, what to cut if time runs short.
"""

def _fmt_chaplain_block(c):
    if not any(c.values()):
        return "(none — generate a generic draft from the profile only; mark all sections as model-filled in the notes.)"
    parts = []
    if c['concept']:       parts.append(f"CORE CONCEPT (chaplain):\n{c['concept']}")
    if c['scripture']:     parts.append(f"SCRIPTURE / SACRED TEXTS (chaplain — authoritative):\n{c['scripture']}")
    if c['illustrations']: parts.append(f"ILLUSTRATIONS / STORIES (chaplain — preserve wording where possible):\n{c['illustrations']}")
    if c['draft']:         parts.append(f"EXISTING DRAFT / NOTES (chaplain — tighten and shape, do not rewrite the voice):\n{c['draft']}")
    if c['constraints']:   parts.append(f"OTHER CONSTRAINTS:\n{c['constraints']}")
    return "\n\n".join(parts)

def build_user_prompt(inp, p, chaplain):
    needs = ', '.join(inp.needs) if inp.needs else '(unspecified)'
    text_pref = inp.text_pref if inp.text_pref else '(see chaplain content below)'
    tradition = inp.tradition if inp.tradition else '(not specified)'
    structure = '\n'.join(p.structure)
    tone = ', '.join(p.tone)
    trauma_posture = '\n'.join(f'- {x}' for x in p.trauma_posture)
    cultural_posture = '\n'.join(f'- {x}' for x in p.cultural_posture)
    avoid = '\n'.join(f'- {x}' for x in p.avoid) if p.avoid else '- (none specified)'
    chaplain_block = _fmt_chaplain_block(chaplain)
    return f"""A) SCENARIO PROFILE
-------------------
Event type: {inp.event_type}{(' / ' + inp.event_other) if inp.event_other else ''}
Broader situation: {inp.situation}
Location: {inp.location}
OPTEMPO: {inp.optempo}

Audience: {inp.audience_primary}
Size: {inp.audience_size}
Trauma level: {inp.trauma}
Fatigue level: {inp.fatigue}
Orientation: {inp.orientation}
Cultural / faith mix: {inp.cultural_mix}

Time cap: {inp.time}
Needs to address: {needs}

Sacred-text preference (form field): {text_pref}
Chaplain tradition (internal reference only): {tradition}

RECOMMENDED FORM
{p.recommended_form}

REQUIRED STRUCTURE & TIMING
{structure}

TONE
{tone}

TRAUMA-RESPONSIVE POSTURE
{trauma_posture}

CULTURAL / PLURALISM POSTURE
{cultural_posture}

AVOID
{avoid}

B) CHAPLAIN CONTENT (AUTHORITATIVE — preserve wording, scripture, illustrations)
--------------------------------------------------------------------------------
{chaplain_block}

TASK
----
Produce a sacred-speech DRAFT following the priority order in your system instructions:
chaplain content first (preserved and shaped), profile second (for structure/tone/posture/time),
your additions only to fill gaps and clearly marked in the chaplain notes.
Honor every hard rule. Output only the Markdown sections specified.
"""

# ----- Form widgets -----
STYLE = {'description_width': '170px'}
LAYOUT = widgets.Layout(width='600px')

w_event_type = widgets.Dropdown(options=['Field devotional','Memorial service','Graveside remarks','Post-trauma debrief / response','Disaster recovery worship','Deployment send-off','Redeployment / homecoming','Daily "word of the day"','Unit resilience / training event','Garrison chapel service','Other'], value='Field devotional', description='Event type:', style=STYLE, layout=LAYOUT)
w_event_other = widgets.Text(description='If Other:', style=STYLE, layout=LAYOUT)
w_situation = widgets.Dropdown(options=['Natural disaster','Combat loss','Training accident','Routine resilience','Deployment cycle','Suicide / line-of-duty death','Other'], value='Other', description='Situation:', style=STYLE, layout=LAYOUT)
w_location = widgets.Dropdown(options=['Field / outdoors','Motor pool','Chapel tent','Disaster site','Briefing room','Garrison chapel','Other'], value='Field / outdoors', description='Location:', style=STYLE, layout=LAYOUT)
w_optempo = widgets.Dropdown(options=['Immediate response','Sustained operations','Recovery phase','Garrison / steady state'], value='Garrison / steady state', description='OPTEMPO:', style=STYLE, layout=LAYOUT)
w_audience_primary = widgets.Dropdown(options=['Soldiers in active recovery / response','Exhausted work crews','Bereaved family / next of kin','Mixed unit formation','Leadership / command team','Families and dependents','Trainees / initial entry','Mixed civilian + military'], value='Mixed unit formation', description='Primary audience:', style=STYLE, layout=LAYOUT)
w_audience_size = widgets.Dropdown(options=['Small (<20)','Medium (20-100)','Large formation (100+)'], value='Small (<20)', description='Audience size:', style=STYLE, layout=LAYOUT)
w_trauma = widgets.Dropdown(options=['Low','Medium','High','Acute / immediate'], value='Low', description='Trauma level:', style=STYLE, layout=LAYOUT)
w_fatigue = widgets.Dropdown(options=['Low','Medium','High','Severe (sustained ops)'], value='Low', description='Fatigue level:', style=STYLE, layout=LAYOUT)
w_orientation = widgets.Dropdown(options=['Mostly emotional','Balanced','Mostly cognitive'], value='Balanced', description='Orientation:', style=STYLE, layout=LAYOUT)
w_cultural_mix = widgets.Dropdown(options=['Homogeneous','Moderately mixed','Highly pluralistic'], value='Highly pluralistic', description='Cultural mix:', style=STYLE, layout=LAYOUT)
w_time = widgets.Dropdown(options=['3-5 min','5-10 min','10-15 min','15-20 min','20+ min'], value='5-10 min', description='Available time:', style=STYLE, layout=LAYOUT)
w_needs = widgets.SelectMultiple(options=['Comfort','Encouragement','Hope','Lament / processing grief','Meaning-making','Resilience','Communal bonding','Individual reflection','Challenge / call to action'], value=('Encouragement','Hope'), description='Needs:', style=STYLE, layout=widgets.Layout(width='600px', height='150px'))
w_tradition = widgets.Text(value='', placeholder='(optional, internal reference)', description='Chaplain tradition:', style=STYLE, layout=LAYOUT)
w_text_pref = widgets.Text(value='', placeholder='(optional, e.g., Psalm 46)', description='Sacred text pref:', style=STYLE, layout=LAYOUT)

form = widgets.VBox([
    widgets.HTML('<h4>Event</h4>'),
    w_event_type, w_event_other, w_situation, w_location, w_optempo,
    widgets.HTML('<h4>Audience</h4>'),
    w_audience_primary, w_audience_size, w_trauma, w_fatigue, w_orientation, w_cultural_mix,
    widgets.HTML('<h4>Time & Goals</h4>'),
    w_time, w_needs,
    widgets.HTML('<h4>Chaplain notes (optional)</h4>'),
    w_tradition, w_text_pref,
])

_AUTOFILL_FIELDS = {
    'event_type': (w_event_type, list(w_event_type.options)),
    'event_other': (w_event_other, None),
    'situation': (w_situation, list(w_situation.options)),
    'location': (w_location, list(w_location.options)),
    'optempo': (w_optempo, list(w_optempo.options)),
    'audience_primary': (w_audience_primary, list(w_audience_primary.options)),
    'audience_size': (w_audience_size, list(w_audience_size.options)),
    'trauma': (w_trauma, list(w_trauma.options)),
    'fatigue': (w_fatigue, list(w_fatigue.options)),
    'orientation': (w_orientation, list(w_orientation.options)),
    'cultural_mix': (w_cultural_mix, list(w_cultural_mix.options)),
    'time': (w_time, list(w_time.options)),
    'text_pref': (w_text_pref, None),
    'tradition': (w_tradition, None),
}
_NEEDS_ALLOWED = list(w_needs.options)

def collect_inputs():
    return ScenarioInputs(
        event_type=w_event_type.value, event_other=w_event_other.value.strip(),
        situation=w_situation.value, location=w_location.value, optempo=w_optempo.value,
        audience_primary=w_audience_primary.value, audience_size=w_audience_size.value,
        trauma=w_trauma.value, fatigue=w_fatigue.value, orientation=w_orientation.value,
        cultural_mix=w_cultural_mix.value, time=w_time.value, needs=list(w_needs.value),
        tradition=w_tradition.value.strip(), text_pref=w_text_pref.value.strip(),
    )

# ----- Chaplain content widgets -----
w_chap_concept = widgets.Textarea(value='', placeholder='One or two sentences: what do you want the audience to walk away with?', description='Core concept:', style={'description_width':'130px'}, layout=widgets.Layout(width='760px', height='80px'))
w_chap_scripture = widgets.Textarea(value='', placeholder='e.g., Psalm 46:1-3 (full text or reference). Include exact text if you want it quoted.', description='Scripture:', style={'description_width':'130px'}, layout=widgets.Layout(width='760px', height='110px'))
w_chap_illustrations = widgets.Textarea(value='', placeholder='Short stories, analogies, military examples. Keep them non-graphic.', description='Illustrations:', style={'description_width':'130px'}, layout=widgets.Layout(width='760px', height='110px'))
w_chap_draft = widgets.Textarea(value='', placeholder='Optional: existing draft text or notes to be tightened and time-fitted.', description='Draft / notes:', style={'description_width':'130px'}, layout=widgets.Layout(width='760px', height='180px'))
w_chap_constraints = widgets.Textarea(value='', placeholder='Optional: things to include or avoid.', description='Constraints:', style={'description_width':'130px'}, layout=widgets.Layout(width='760px', height='70px'))

def collect_chaplain_content():
    return {'concept': w_chap_concept.value.strip(),
            'scripture': w_chap_scripture.value.strip(),
            'illustrations': w_chap_illustrations.value.strip(),
            'draft': w_chap_draft.value.strip(),
            'constraints': w_chap_constraints.value.strip()}

print('Setup complete. Run the next cell to launch the wizard.')


In [ ]:
# Sacred Speech Planner v4 — wizard launcher
# Run this AFTER the setup cell above.

state = {'inputs': None, 'profile': None, 'plan_md': None, 'draft': None, 'chaplain': {}}

# ---------- Screen 1: Welcome + key check ----------
def _key_status_html():
    if not API_KEY:
        return ('<div style="color:#a00;padding:12px;background:#fff0f0;border-left:4px solid #a00;border-radius:4px">'
                '<b>X No API key found.</b><br>Add <code>OPENAI_API_KEY</code> to Colab Secrets '
                '(key icon in left sidebar), then click <b>Runtime &rarr; Restart and run all</b>.</div>')
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{'role':'system','content':'Reply with the single word: ready.'},
                      {'role':'user','content':'ping'}],
            max_tokens=5, temperature=0)
        reply = (resp.choices[0].message.content or '').strip()
        return (f'<div style="color:#070;padding:12px;background:#f0fff0;border-left:4px solid #070;border-radius:4px">'
                f'<b>OK API key works.</b> Model <code>{MODEL}</code> replied: <code>{reply!r}</code>.<br>'
                f'Click <b>Next &rarr;</b> to begin.</div>')
    except Exception as e:
        return (f'<div style="color:#a60;padding:12px;background:#fff8e6;border-left:4px solid #a60;border-radius:4px">'
                f'<b>! Key loaded but test call failed:</b> {e}<br>'
                f'Common causes: invalid key, no billing, wrong PROVIDER, or model not available to your key.</div>')

screen1 = widgets.VBox([
    widgets.HTML('<h2 style="margin:0">Welcome</h2>'
                 '<p>This wizard walks you through 5 short screens to plan a sacred-speech moment for any Army scenario.</p>'),
    widgets.HTML(value=_key_status_html()),
    widgets.HTML('<h3>OPSEC reminder</h3>'
                 '<p>Use <b>generic descriptors only</b> &mdash; no PII, names, unit identifiers, or sensitive operational detail. '
                 'Your inputs in screens 2 and 4 are sent to the OpenAI API.</p>'
                 '<h3>Pluralism</h3>'
                 '<p>Descriptive and culturally adaptive. Does not prescribe theology. You supply all sacred-text content.</p>'),
])

# ---------- Screen 2: Scenario description + Autofill ----------
nl_input = widgets.Textarea(
    value='', placeholder='Describe the event, audience, time available, environment, fatigue/trauma indicators, and any constraints. Generic descriptors only -- no PII.',
    description='Scenario:', style={'description_width':'90px'},
    layout=widgets.Layout(width='760px', height='180px'))
autofill_btn = widgets.Button(description='Autofill from description', button_style='info', icon='magic')
skip_autofill_btn = widgets.Button(description='Skip - fill form manually', icon='forward')
autofill_status = widgets.Output()

def on_autofill(_):
    with autofill_status:
        clear_output()
        if not API_KEY:
            print('No API key. Go back to screen 1.')
            return
        desc = nl_input.value.strip()
        if not desc:
            print('Type a scenario description first, or click "Skip" to fill the form manually.')
            return
        print(f'Calling {MODEL} for scenario extraction...')
        try:
            raw = llm_chat(AUTOFILL_SYSTEM, desc, temperature=0.1, response_format={'type':'json_object'})
        except Exception as e:
            print(f'LLM call failed: {e}')
            return
        try:
            data = json.loads(raw)
        except Exception as e:
            print(f'Could not parse JSON: {e}\n\nRaw output:\n{raw}')
            return
        for key, (widget, allowed) in _AUTOFILL_FIELDS.items():
            if key not in data: continue
            v = data[key]
            if allowed is not None and v not in allowed:
                continue
            widget.value = v
        needs = data.get('needs', []) or []
        needs = [n for n in needs if n in _NEEDS_ALLOWED]
        if needs:
            w_needs.value = tuple(needs)
        clear_output()
        print('Autofill applied. Click Next to review the form.')
        if data.get('rationale'):
            display(Markdown('**LLM rationale:**\n\n' + str(data['rationale'])))

def on_skip(_):
    with autofill_status:
        clear_output()
        print('Skipping autofill. Click Next to fill the form manually.')

autofill_btn.on_click(on_autofill)
skip_autofill_btn.on_click(on_skip)

screen2 = widgets.VBox([
    widgets.HTML('<h2 style="margin:0">1. Describe the scenario</h2>'
                 '<p>Type or paste plain language. Click <b>Autofill</b> and the AI will populate the form for the next screen. '
                 'You will review and edit there. Or skip to fill the form manually.</p>'),
    nl_input,
    widgets.HBox([autofill_btn, skip_autofill_btn]),
    autofill_status,
])

# ---------- Screen 3: Review form ----------
screen3 = widgets.VBox([
    widgets.HTML('<h2 style="margin:0">2. Review and edit</h2>'
                 '<p>Tweak any fields the AI got wrong, especially <b>Time</b>, <b>Trauma level</b>, '
                 '<b>Fatigue level</b>, and <b>Cultural mix</b> &mdash; these drive the recommended form and tone.</p>'),
    form,
])

# ---------- Screen 4: Plan + chaplain content ----------
plan_out = widgets.Output()

def regenerate_plan():
    with plan_out:
        clear_output()
        try:
            inp = collect_inputs()
            p = profile(inp)
            md = render_profile(p, inp)
            state['inputs'], state['profile'], state['plan_md'] = inp, p, md
            display(Markdown(md))
        except Exception as e:
            print(f'Could not generate plan: {e}')

regen_btn = widgets.Button(description='Regenerate plan', icon='refresh')
regen_btn.on_click(lambda _: regenerate_plan())

chaplain_block = widgets.VBox([
    widgets.HTML('<h3 style="margin-top:24px">(Optional) Add your own concept, scripture, illustrations</h3>'
                 '<p style="color:#555;font-size:90%">Anything you provide becomes the <b>authoritative source</b> for the AI draft on the next screen &mdash; '
                 'your scripture choices, illustrations, and core message will be preserved, not overruled.</p>'),
    w_chap_concept, w_chap_scripture, w_chap_illustrations, w_chap_draft, w_chap_constraints,
])

screen4 = widgets.VBox([
    widgets.HTML('<h2 style="margin:0">3. Generated plan</h2>'
                 '<p>Rules-based profile + organizer template based on your form. '
                 'If you change form fields and come back here, click <b>Regenerate plan</b>.</p>'),
    regen_btn,
    plan_out,
    chaplain_block,
])

# ---------- Screen 5: AI Draft ----------
draft_out = widgets.Output()
gen_draft_btn = widgets.Button(description='Generate AI Draft', button_style='success', icon='magic')
export_btn = widgets.Button(description='Export plan + draft to Markdown', icon='download')
export_status = widgets.Output()

def on_gen_draft(_):
    with draft_out:
        clear_output()
        if state.get('profile') is None:
            print('Generate a plan on screen 3 first (go Back, then Next).')
            return
        if not API_KEY:
            print('No API key. Go back to screen 1.')
            return
        chaplain = collect_chaplain_content()
        state['chaplain'] = chaplain
        provided = [k for k, v in chaplain.items() if v]
        print(f'Calling {MODEL}...')
        if provided:
            print(f'Chaplain content: {", ".join(provided)}')
        else:
            print('No chaplain content provided -- draft will be generic.')
        try:
            user_prompt = build_user_prompt(state['inputs'], state['profile'], chaplain)
            draft = llm_chat(DRAFT_SYSTEM_PROMPT, user_prompt, temperature=0.6)
        except Exception as e:
            print(f'LLM call failed: {e}')
            return
        state['draft'] = draft
        clear_output()
        display(Markdown('## AI Draft (chaplain edits before use)\n\n' + draft))
        display(Markdown('---\n*Draft only. Verify scripture quotations, pluralism fit, trauma posture, and time discipline before delivery.*'))

def on_export(_):
    with export_status:
        clear_output()
        if state.get('plan_md') is None:
            print('No plan generated yet.')
            return
        stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        fname = f'sacred_speech_plan_{stamp}.md'
        try:
            with open(fname, 'w', encoding='utf-8') as f:
                f.write(f'# Sacred Speech Plan ({stamp})\n\n')
                f.write(state['plan_md'])
                ch = state.get('chaplain') or {}
                if any(ch.values()):
                    f.write('\n\n## Chaplain Content (source material)\n')
                    for label, key in [('Core concept','concept'),('Scripture','scripture'),
                                       ('Illustrations','illustrations'),('Draft / notes','draft'),
                                       ('Constraints','constraints')]:
                        if ch.get(key):
                            f.write(f'\n**{label}:**\n\n{ch[key]}\n')
                if state.get('draft'):
                    f.write('\n\n## AI Draft (chaplain edits before use)\n\n')
                    f.write(state['draft'])
                    f.write('\n\n---\n*Draft only.*\n')
            print(f'Saved: {os.path.abspath(fname)}')
            print('In Colab, open the Files panel (folder icon, left sidebar) to download.')
        except Exception as e:
            print(f'Save failed: {e}')

gen_draft_btn.on_click(on_gen_draft)
export_btn.on_click(on_export)

screen5 = widgets.VBox([
    widgets.HTML('<h2 style="margin:0">4. AI draft</h2>'
                 '<p>Combines your content (authoritative) with the plan (shaping). '
                 'Edit the result before delivery.</p>'),
    gen_draft_btn,
    draft_out,
    widgets.HTML('<h3 style="margin-top:24px">Export</h3>'),
    export_btn,
    export_status,
])

# ---------- Stepper chrome ----------
SCREENS = [screen1, screen2, screen3, screen4, screen5]
TITLES = ['Welcome', 'Describe', 'Review', 'Plan', 'Draft']
N = len(SCREENS)

stack = widgets.Stack(children=SCREENS, selected_index=0)
progress = widgets.IntProgress(value=1, min=0, max=N, description='', layout=widgets.Layout(width='320px'))
title_lbl = widgets.HTML()
back_btn = widgets.Button(description='< Back', disabled=True)
next_btn = widgets.Button(description='Next >', button_style='primary')

def update_chrome():
    i = stack.selected_index
    progress.value = i + 1
    title_lbl.value = (f'<div style="font-size:13px;color:#666;margin-bottom:2px">'
                       f'Screen {i+1} of {N}</div>'
                       f'<h2 style="margin:0">{TITLES[i]}</h2>')
    back_btn.disabled = (i == 0)
    next_btn.disabled = (i == N - 1)
    next_btn.description = 'Done' if i == N - 1 else 'Next >'

def go(delta):
    new_i = stack.selected_index + delta
    if new_i < 0 or new_i >= N:
        return
    if new_i == 3 and delta > 0:
        regenerate_plan()
    stack.selected_index = new_i
    update_chrome()

back_btn.on_click(lambda _: go(-1))
next_btn.on_click(lambda _: go(1))

chrome = widgets.HBox([back_btn, next_btn, widgets.HTML('&nbsp;&nbsp;&nbsp;'), progress])
wizard = widgets.VBox([title_lbl, chrome,
                       widgets.HTML('<hr style="margin:8px 0">'),
                       stack])

update_chrome()
display(wizard)
